# S1 segmentation and initial-state walkthrough

This notebook begins with management-unit segmentation, then carries either the faithful LETO baseline or the boundary-overlay baseline through the same TreeMap/FIA attribution and FVS initial-state stages. The production transformations live in `pipeline/s1_initial_state`; cells here select and inspect them without reimplementing them.

## Inputs and preflight

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd
import rasterio

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.s1_initial_state.data_sources import (  # noqa: E402
    ProductionDataPaths,
    load_fia_trees_sqlite,
    load_treemap_lookup,
    preflight_production_data,
)
from pipeline.s1_initial_state.leto_initial_state import (  # noqa: E402
    build_initial_state,
    build_management_unit_crosswalk,
    filter_and_normalize_weights,
    load_species_lookup,
    prepare_direct_tree_rows,
    write_initial_state,
)
from pipeline.s1_initial_state.segmentation import boundary_overlay  # noqa: E402
from pipeline.s1_initial_state.segmentation.comparison import (  # noqa: E402
    compare_attribution,
    compare_segmentations,
)
from pipeline.s1_initial_state.segmentation.leto import (  # noqa: E402
    LetoSegmentationConfig,
    assign_majority_ownership,
    assign_smz_percent,
    build_leto_management_units,
)
from pipeline.s1_initial_state.weights import build_plot_weights  # noqa: E402

In [ ]:
PRODUCTION_DATA_ROOT = Path("/mnt/d")
SEGMENTATION_METHOD = "leto"  # choose 'leto' or 'boundary_overlay'
SEGMENTATION_METHODS = ("leto", "boundary_overlay")
COUNTY_FIPS = "125"
COUNTY_NAME = {
    "003": "BAKER",
    "023": "COLUMBIA",
    "047": "HAMILTON",
    "089": "NASSAU",
    "125": "UNION",
}[COUNTY_FIPS]
BOUNDARY_PROCESS_OUTPUT_DIR = (
    PROJECT_ROOT / "data/interim/management_units/boundary_overlay_process"
)
BASELINE_OUTPUT_DIR = PROJECT_ROOT / "data/interim/management_units/baselines"
BASELINE_UNITS_PATHS = {
    "leto": BASELINE_OUTPUT_DIR / "leto" / f"12{COUNTY_FIPS}" / "ManagementUnits.gpkg",
    "boundary_overlay": BASELINE_OUTPUT_DIR
    / "boundary_overlay"
    / f"12{COUNTY_FIPS}"
    / "ManagementUnits.gpkg",
}
BMP_RULES_PATH = PROJECT_ROOT / "config/bmp_rules.yaml"
OUTPUT_DIR = PROJECT_ROOT / "data/interim/fvs" / SEGMENTATION_METHOD
MIN_PLT_WEIGHT = 0.05
WRITE_OUTPUTS = False

if SEGMENTATION_METHOD not in SEGMENTATION_METHODS:
    raise ValueError(f"SEGMENTATION_METHOD must be one of {SEGMENTATION_METHODS}")
sources = ProductionDataPaths.from_root(PRODUCTION_DATA_ROOT)
preflight_production_data(sources)
if SEGMENTATION_METHOD == "boundary_overlay":
    boundary_overlay.preflight_boundary_overlay_data(sources.root, BMP_RULES_PATH)
source_preflight = pd.DataFrame(
    [
        {"input": name, "path": str(path), "exists": path.exists()}
        for name, path in asdict(sources).items()
        if name != "root"
    ]
)
display(source_preflight)

## Management-unit segmentation

Segmentation is the first analytical stage. `SEGMENTATION_METHOD` selects the faithful LETO implementation or the canonical boundary-overlay implementation. With `WRITE_OUTPUTS = True`, the selected result is written to `baselines/<method>/<county>/ManagementUnits.gpkg`. To create a comparison pair, run the notebook once with `leto`, then once with `boundary_overlay`; return `WRITE_OUTPUTS` to `False` for read-only comparison. Boundary-overlay selection in read-only mode consumes its existing canonical GeoPackage and fails closed when that artifact is absent.

In [ ]:
LETO_CONFIG = LetoSegmentationConfig(
    max_acres=200.0,
    acres_per_point=100.0,
    min_distance_feet=1_000.0,
    min_acres=5.0,
    smz_buffer_feet=35.0,
    seed=0,
)
BOUNDARY_OVERLAY_PARAMETERS = {
    "county_fips": COUNTY_FIPS,
    "split_large": True,
    "save_qa": False,
    "overwrite": True,
}
selected_parameters = (
    asdict(LETO_CONFIG)
    if SEGMENTATION_METHOD == "leto"
    else BOUNDARY_OVERLAY_PARAMETERS
)
display(pd.Series(selected_parameters, name=SEGMENTATION_METHOD))

treemap_lookup = load_treemap_lookup(sources.treemap_vat)
parcels = gpd.read_file(sources.parcels, layer="FL_5_Co_Parcels")
if "CNTYNAME" not in parcels:
    raise ValueError(
        "Production parcels must contain CNTYNAME for paired AOI selection"
    )
parcels = parcels.loc[parcels["CNTYNAME"] == COUNTY_NAME].copy()
if parcels.empty:
    raise ValueError(f"No production parcels found for {COUNTY_NAME} ({COUNTY_FIPS})")
streams = gpd.read_file(f"zip://{sources.streams}")
if SEGMENTATION_METHOD == "leto":
    management_units, plot_weights = build_leto_management_units(
        sources.treemap,
        treemap_lookup,
        parcels,
        sources.ownership,
        streams,
        LETO_CONFIG,
    )
else:
    if WRITE_OUTPUTS:
        boundary_overlay.process_county(
            output_dir=BOUNDARY_PROCESS_OUTPUT_DIR,
            data_root=sources.root,
            config_path=BMP_RULES_PATH,
            dry_run=False,
            **BOUNDARY_OVERLAY_PARAMETERS,
        )
        boundary_units_path = (
            BOUNDARY_PROCESS_OUTPUT_DIR
            / f"12{COUNTY_FIPS}"
            / "candidate_management_units.gpkg"
        )
    else:
        boundary_units_path = BASELINE_UNITS_PATHS["boundary_overlay"]
        if not boundary_units_path.exists():
            raise FileNotFoundError(
                f"{boundary_units_path} is absent; set WRITE_OUTPUTS = True and run "
                "the boundary_overlay strategy to create it."
            )
    management_units = gpd.read_file(boundary_units_path)
    plot_weights = build_plot_weights(management_units, sources.treemap, treemap_lookup)
    management_units = assign_majority_ownership(management_units, sources.ownership)
    management_units = assign_smz_percent(
        management_units, streams, LETO_CONFIG.smz_buffer_feet
    )

canonical_units_path = BASELINE_UNITS_PATHS[SEGMENTATION_METHOD]
if WRITE_OUTPUTS:
    canonical_units_path.parent.mkdir(parents=True, exist_ok=True)
    management_units.to_file(
        canonical_units_path, layer="management_units", driver="GPKG"
    )
    print(f"Canonical baseline written: {canonical_units_path}")

segmentation_diagnostics = pd.Series(
    {
        "method": SEGMENTATION_METHOD,
        "unit_count": len(management_units),
        "total_acres": management_units["Acres"].sum(),
        "median_acres": management_units["Acres"].median(),
        "slivers_below_5_acres": (management_units["Acres"] < 5.0).sum(),
        "units_above_200_acres": (management_units["Acres"] > 200.0).sum(),
    }
)
display(segmentation_diagnostics.to_frame(name="value"))
if SEGMENTATION_METHOD == "boundary_overlay" and "size_class" in management_units:
    display(management_units["size_class"].value_counts().rename("units"))
management_units.plot(column="Acres", figsize=(8, 8), legend=True)

## Management units and TreeMap alignment

Both segmentation methods satisfy the shared S1 management-unit contract. TreeMap attribution uses the native raster grid and the portable pixel-center rule.

In [ ]:
with rasterio.open(sources.treemap) as source:
    treemap_metadata = pd.Series(
        {
            "crs": str(source.crs),
            "pixel_width": source.transform.a,
            "pixel_height": source.transform.e,
            "nodata": source.nodata,
            "width": source.width,
            "height": source.height,
        }
    )
display(management_units.drop(columns="geometry").head())
display(treemap_metadata)
display(management_units["SEGMENTATION_METHOD"].value_counts())

## MU x PLT_CN weights

This is the non-ArcPy counterpart to `assign_plt_cn` in `LETO.V1.1.txt`.

In [ ]:
weight_diagnostics = plot_weights.groupby("MU_ID").agg(
    donor_plots=("PLT_CN", "size"),
    valid_cells=("CELL_COUNT", "sum"),
    raw_weight_sum=("WEIGHT", "sum"),
)
display(plot_weights.head())
display(weight_diagnostics.describe())

## Segmentation baseline comparison

When the counterpart baseline GeoPackage is available, compare both methods over the same study inputs. Otherwise the notebook reports a structured `counterpart_unavailable` status and the exact path to create. These are descriptive diagnostics; they do not establish management realism or select a hybrid.

In [ ]:
counterpart_method = next(
    method for method in SEGMENTATION_METHODS if method != SEGMENTATION_METHOD
)
counterpart_path = BASELINE_UNITS_PATHS[counterpart_method]
if counterpart_path.exists():
    counterpart_units = gpd.read_file(counterpart_path)
    counterpart_weights = build_plot_weights(
        counterpart_units, sources.treemap, treemap_lookup
    )
    baseline_units = {
        SEGMENTATION_METHOD: management_units,
        counterpart_method: counterpart_units,
    }
    baseline_weights = {
        SEGMENTATION_METHOD: plot_weights,
        counterpart_method: counterpart_weights,
    }
    segmentation_comparison = compare_segmentations(
        baseline_units["leto"],
        baseline_units["boundary_overlay"],
        reference_name="leto",
        candidate_name="boundary_overlay",
    )
    attribution_comparison = compare_attribution(
        baseline_weights["leto"], baseline_weights["boundary_overlay"]
    )
    display(segmentation_comparison.to_frame(name="value"))
    display(attribution_comparison.to_frame(name="value"))
else:
    comparison_status = pd.Series(
        {
            "status": "counterpart_unavailable",
            "counterpart_method": counterpart_method,
            "expected_path": str(counterpart_path),
            "action": (
                f"Set SEGMENTATION_METHOD = '{counterpart_method}' and "
                "WRITE_OUTPUTS = True, then run the notebook."
            ),
        },
        name="comparison_status",
    )
    display(comparison_status.to_frame(name="value"))

## FIA join coverage

LETO filters donors below five percent, renormalizes within each management unit, and joins multistate FIA tree files because TreeMap can borrow plots across state lines.

In [ ]:
crosswalk = build_management_unit_crosswalk(management_units, plot_weights)
normalized_weights = filter_and_normalize_weights(
    plot_weights, crosswalk, min_plot_weight=MIN_PLT_WEIGHT
)
weighted_plots = set(normalized_weights["PLT_CN"])
fia_trees = load_fia_trees_sqlite(sources.fiadb, weighted_plots)
fia_plots = set(fia_trees["PLT_CN"])
join_coverage = pd.Series(
    {
        "weighted plots": len(weighted_plots),
        "matched FIA plots": len(weighted_plots & fia_plots),
        "unmatched FIA plots": len(weighted_plots - fia_plots),
    }
)
display(join_coverage)
display(pd.Series(sorted(weighted_plots - fia_plots), name="unmatched_PLT_CN").head(25))

## Species and live-tree preparation

In [ ]:
species_lookup = load_species_lookup(
    sources.species_crosswalk, "EasternSpeciesTranslator"
)
direct_trees = prepare_direct_tree_rows(normalized_weights, fia_trees, species_lookup)
tree_diagnostics = pd.Series(
    {
        "combined FIA rows": len(fia_trees),
        "direct FVS tree rows": len(direct_trees),
        "direct runnable stands": direct_trees["STAND_ID"].nunique(),
        "translated FVS species": direct_trees["SPECIES"].nunique(),
    }
)
display(tree_diagnostics)
display(direct_trees.head())

## Nearest-runnable-unit imputation

Units without usable live trees receive the nearest runnable polygon's weighted tree list. `TREE_SOURCE`, `DONOR_STAND_ID`, and `NEAR_DIST` preserve that provenance.

In [ ]:
tables = build_initial_state(
    management_units,
    plot_weights,
    fia_trees,
    species_lookup,
    min_plot_weight=MIN_PLT_WEIGHT,
)
display(tables.diagnostics.to_frame())
source_summary = tables.trees.groupby("TREE_SOURCE").agg(
    tree_rows=("TREE_ID", "size"),
    stands=("STAND_ID", "nunique"),
)
display(source_summary)
display(
    tables.trees.loc[
        tables.trees["TREE_SOURCE"] == "IMPUTED_NEAREST",
        ["STAND_ID", "DONOR_STAND_ID", "NEAR_DIST"],
    ]
    .drop_duplicates()
    .head(25)
)

## Initial-state map

In [ ]:
direct_ids = set(
    tables.trees.loc[tables.trees["TREE_SOURCE"] == "FIA_WEIGHTED_DIRECT", "MU_ID"]
)
imputed_ids = set(
    tables.trees.loc[tables.trees["TREE_SOURCE"] == "IMPUTED_NEAREST", "MU_ID"]
)
state_map = management_units.copy()
state_map["MU_ID"] = state_map["MU_ID"].astype("string")
state_map["INITIAL_STATE"] = "missing"
state_map.loc[state_map["MU_ID"].isin(imputed_ids), "INITIAL_STATE"] = "imputed_nearest"
state_map.loc[state_map["MU_ID"].isin(direct_ids), "INITIAL_STATE"] = (
    "fia_weighted_direct"
)
display(state_map["INITIAL_STATE"].value_counts())
state_map.plot(column="INITIAL_STATE", categorical=True, legend=True, figsize=(10, 10))

## Write LETO-compatible outputs

In [ ]:
if WRITE_OUTPUTS:
    output_paths = write_initial_state(tables, OUTPUT_DIR)
    display(pd.Series({name: str(path) for name, path in output_paths.items()}))
else:
    print("WRITE_OUTPUTS is False; inspected results were not written.")